# LAB- P1: El Modelo IS-LM Dinámico en Tiempo Continuo
**Proyecto MACRO-AI-COMP (Convocatoria INNOVA26, UMA / Banco Santander)**
*   **Código de LAB-**: LAB-P1-v1.0
*   **Capítulo de Referencia**: Capítulo 2, *An Introduction to Computational Macroeconomics* (Bongers, Gómez y Torres, Vernon Press, 2019)
*   **Autores**: Equipo Docente MACRO-AI-COMP
*   **Objetivo**: Analizar la respuesta dinámica y la transición temporal de una economía IS-LM cerrada ante shocks de políticas macroeconómicas (monetarias y fiscales) bajo un ajuste gradual de la producción e inflación por curva de Phillips.

---

## Objetivos de Aprendizaje
Al finalizar esta práctica, serás capaz de:
1.  **Comprender** el mecanismo de transmisión dinámica de un shock monetario y fiscal en un modelo de equilibrio general simple de corto-medio plazo.
2.  **Visualizar** el principio de *neutralidad del dinero* a largo plazo y la no neutralidad a corto plazo.
3.  **Aprender** a integrar sistemas de ecuaciones diferenciales ordinarias (ODEs) en Python utilizando `scipy.integrate.solve_ivp`.
4.  **Evaluar** críticamente los resultados simulados comparándolos contra el "oráculo" analítico y los apéndices numéricos del libro.
 (Julia)

> **👋 BIENVENIDA A LA PRÁCTICA - LEER ANTES DE EMPEZAR**
> 
> *   **¿Nunca has usado Jupyter?** No te preocupes. Este cuaderno es interactivo. Haz clic en cualquier celda de código y pulsa **`Shift + Enter`** para ejecutarla. Ve de arriba a abajo en orden.
> *   **¿Se ha congelado o sale un asterisco `[*]` eterno?** Ve al menú superior y dale a `Kernel` ➔ `Restart`.
> *   **El objetivo** de esta práctica es que juegues con la economía. Cambia los números del código que representan impuestos, dinero o tecnología, vuelve a ejecutar y mira los gráficos. ¡No puedes romper nada!
>

### 🕹️ GUÍA RÁPIDA PARA DUMMIES - IS-LM Dinámico
*   **¿Qué estamos haciendo aquí?** Simulamos una economía cerrada. El tipo de interés y la producción se ajustan según la oferta y demanda de bienes y dinero.
*   **Controles clave para cambiar:** 
    *   `m0`: La oferta de dinero (política monetaria).
    *   `beta0`: El gasto del gobierno (política fiscal).
*   **El Shock:** Si subes `m0` (imprimir dinero), verás que a corto plazo el PIB sube y los intereses bajan. Pero a largo plazo, el PIB vuelve a su sitio y solo suben los precios (neutralidad del dinero).
*   **¡Prueba esto!** Usa los deslizadores interactivos al final del cuaderno para arrastrar y ver la animación del shock en vivo.


In [1]:
# En Google Colab se activarían y descargarían los paquetes necesarios.
# using Pkg; Pkg.activate("."); Pkg.instantiate()


In [2]:
using Pkg
Pkg.activate("../..")

using MacroAIComp
using Plots
import Plots: mm
using LinearAlgebra
using DifferentialEquations
using Interact
using BenchmarkTools


  Activating project at `C:\Users\AntonioRC\Desktop\PIE`


WebIO._IJuliaInit()

## 1. El Marco Teórico: Ecuaciones y Parámetros

El modelo IS-LM dinámico en tiempo continuo describe una economía cerrada con las siguientes relaciones estructurales:

### 1.1 Ecuaciones de Comportamiento

1.  **Mercado Monetario (Curva LM):** El tipo de interés nominal ($i$) equilibra la oferta real de dinero con la demanda de saldos reales:
    $$M - P = \psi Y - \theta i$$
    Donde:
    *   $M$: Logaritmo de la oferta monetaria nominal (controlada por el Banco Central).
    *   $P$: Logaritmo del nivel de precios.
    *   $Y$: Nivel de producción real (renta).
    *   $\psi$: Sensibilidad de la demanda de dinero respecto a la renta.
    *   $\theta$: Sensibilidad de la demanda de dinero respecto al tipo de interés nominal.

    Al despejar el tipo de interés nominal ($i$), obtenemos el equilibrio del mercado monetario en función del estado de la economía ($Y, P$):
    $$i(Y, P) = \frac{P - M + \psi Y}{\theta}$$

2.  **Demanda Agregada (Curva IS):** La demanda agregada planeada ($Y^d$) depende negativamente del tipo de interés real ($r = i - \pi$):
    $$Y^d = \beta_0 - \beta_1(i - \pi)$$
    Donde:
    *   $\beta_0$: Demanda agregada autónoma (gasto público, consumo autónomo).
    *   $\beta_1$: Sensibilidad de la inversión y consumo respecto al tipo de interés real.
    *   $\pi = \frac{dP}{dt}$: Tasa de inflación observada en tiempo continuo.

3.  **Curva de Phillips (Dinámica de Precios):** La tasa de inflación responde a la presión de demanda (brecha de producción):
    $$\frac{dP}{dt} = \mu(Y - \bar{Y})$$
    Donde:
    *   $\bar{Y}$ (`ypot0`): Nivel de producto potencial o pleno empleo de largo plazo.
    *   $\mu$: Sensibilidad de la inflación respecto a la brecha de producción (ajuste de precios).

4.  **Ajuste del Producto (Dinámica de Producción):** La producción real ($Y$) se ajusta gradualmente hacia el nivel de demanda agregada:
    $$\frac{dY}{dt} = \nu(Y^d - Y)$$
    Donde:
    *   $\nu$: Velocidad de ajuste de la producción física frente al exceso de demanda.

---

### 1.2 Reducción a un Sistema Dinámico de Ecuaciones Diferenciales Ordinarias (ODEs)

Para simular la economía en el tiempo, reducimos las cuatro relaciones anteriores a un sistema de dos ecuaciones diferenciales ordinarias (ODEs) con dos variables de estado: la producción real ($Y$) y el nivel de precios ($P$).

Sustituyendo el tipo de interés nominal $i$ y la tasa de inflación $\frac{dP}{dt}$ en la demanda agregada $Y^d$, obtenemos:
$$Y^d = \beta_0 - \beta_1 \left( \frac{P - M + \psi Y}{\theta} - \mu(Y - \bar{Y}) \right)$$

Sustituyendo esta demanda agregada en la ecuación de ajuste de la producción, el sistema de dos variables de estado queda completamente determinado por:
$$\frac{dY}{dt} = \nu \left[ \beta_0 - \beta_1 \left( \frac{P - M + \psi Y}{\theta} - \mu(Y - \bar{Y}) \right) - Y \right]$$
$$\frac{dP}{dt} = \mu(Y - \bar{Y})$$

Este es el sistema continuo bidimensional que resolveremos numéricamente. Las variables $Y$ y $P$ se comportan como variables de estado lentas (continuas), mientras que $i$ y $Y^d$ son variables algebraicas que pueden dar saltos discretos e instantáneos cuando ocurre una política económica imprevista.


In [3]:
params = default_calibration(ISLMParams)
println(params)


ISLMParams(0.5, 0.01, 50.0, 0.01, 0.2, 2100.0, 100.0, 2000.0)


## 2. Equilibrio de Largo Plazo (Estado Estacionario)

En el largo plazo, todas las variables dinámicas se estabilizan. Esto significa que las derivadas temporales son nulas:
$$\frac{dY}{dt} = 0 \quad \text{y} \quad \frac{dP}{dt} = 0$$

A partir de estas dos condiciones de equilibrio de largo plazo, podemos deducir analíticamente los valores de estado estacionario ($Y^*, P^*, i^*$):

1.  **De la Curva de Phillips:** $\frac{dP}{dt} = \mu(Y^* - \bar{Y}) = 0 \Rightarrow Y^* = \bar{Y}$.
    *   *Significado:* La producción de largo plazo coincide necesariamente con el **producto potencial** ($2000.0$).
2.  **Del Ajuste de Producción:** $\frac{dY}{dt} = \nu(Y^d - Y^*) = 0 \Rightarrow Y^d = Y^* = \bar{Y}$.
    *   *Significado:* La demanda agregada planeada iguala a la producción en el largo plazo.
3.  **De la Curva IS:** Dado que la tasa de inflación es nula en estado estacionario ($\pi^* = 0$), el tipo de interés real es igual al tipo de interés nominal ($r^* = i^*$). Por tanto:
    $$\bar{Y} = \beta_0 - \beta_1 i^* \Rightarrow i^* = \frac{\beta_0 - \bar{Y}}{\beta_1}$$
    *   *Cálculo:* $i^* = \frac{2100.0 - 2000.0}{50.0} = 2.0\%$ (0.02).
4.  **De la Curva LM:** Despejando el nivel de precios de equilibrio $P^*$:
    $$P^* = \theta i^* + M_0 - \psi \bar{Y}$$
    *   *Cálculo:* $P^* = 0.5 \cdot 2.0 + 100.0 - 0.01 \cdot 2000.0 = 1.0 + 100.0 - 20.0 = 81.0$.

A continuación, ejecutamos la función de cálculo numérico para confirmar que obtenemos los mismos resultados.


In [4]:
ss = steady_state(params)

println("VALORES DE EQUILIBRIO DE LARGO PLAZO:")
println("  Renta de pleno empleo (Y*) : ", ss["Y"])
println("  Nivel de precios (P*)      : ", ss["P"])
println("  Tipo de interés (i*)       : ", round(ss["i"], digits=2), "%")
println("  Demanda agregada (Yd*)     : ", ss["Yd"])


VALORES DE EQUILIBRIO DE LARGO PLAZO:
  Renta de pleno empleo (Y*) : 2000.0
  Nivel de precios (P*)      : 81.0
  Tipo de interés (i*)       : 2.0%
  Demanda agregada (Yd*)     : 2000.0


## 3. Detrás de la Escena: Resolución Numérica de ODEs en Julia

Para simular una economía dinámica en tiempo continuo, debemos resolver el sistema de ecuaciones diferenciales:
$$\frac{d\mathbf{x}}{dt} = \mathbf{f}(t, \mathbf{x}, \mathbf{\theta})$$
Donde el vector de estado es $\mathbf{x}(t) = [Y(t), P(t)]^T$ y los parámetros son $\mathbf{\theta}$. En Julia, utilizamos la función científica `DifferentialEquations.jl`. 

### ¿Cómo funciona `solve_ivp`?
1. **El Algoritmo de Integración:** Por defecto, utiliza el método `'RK45'` (Runge-Kutta de orden 4 con estimación de error de orden 5). Este algoritmo evalúa el campo vectorial (las derivadas) en múltiples puntos de cada subintervalo temporal y ajusta el tamaño del paso dinámicamente para garantizar que la solución numérica no se desvíe de la trayectoria teórica.
2. **Definición de la función de derivadas:** Requiere que le proporcionemos una función que tome el tiempo actual `t` y el estado actual `state` y devuelva el vector de derivadas `[dY/dt, dP/dt]`. Aunque el tiempo no aparece explícitamente en el lado derecho de nuestras ecuaciones (es un sistema autónomo), `solve_ivp` requiere obligatoriamente que `t` sea el primer argumento de la función de derivadas.
3. **El Intervalo Temporal (`t_span`):** Define desde qué instante hasta qué instante simulamos la transición (por ejemplo, de $t=0$ a $t=30$ períodos).
4. **Condiciones Iniciales (`y0`):** La posición de partida del sistema en el instante $t=0$ (en este caso, los valores de producción y precios en el estado estacionario inicial pre-shock).

A continuación se muestra de forma explícita cómo programar la dinámica del modelo utilizando la biblioteca estándar y los parámetros cargados. De este modo puedes ver la lógica que opera dentro de la biblioteca `macroaicomp`:


In [5]:
# Así definimos el sistema dinámico en Julia (ver src/models/ISLM.jl)
function custom_system_dynamics!(du, u, p, t)
    Y = u[1]
    P = u[2]
    
    params = p
    # Curva LM
    i_rate = (P - params.m0 + params.psi * Y) / params.theta
    # Curva IS
    Y_d = params.beta0 - params.beta1 * (i_rate - 0.0) # pi=0
    
    # Phillips y Ajuste
    du[1] = params.ni * (Y_d - Y)
    du[2] = params.mi * (Y - params.ypot0)
end


custom_system_dynamics! (generic function with 1 method)

## 4. Transición Dinámica y Shocks de Política (Simulación Interactiva)

### 4.1 El Mecanismo de Transmisión Económica
Cuando se produce un **shock monetario expansivo** ($M_0 \uparrow$):
1.  **Impacto inmediato (Corto Plazo):** Los precios ($P$) son rígidos en el instante inicial, por lo que la oferta de dinero real ($M - P$) se expande. Esto desplaza la curva LM hacia abajo, haciendo que el tipo de interés ($i$) caiga de forma abrupta.
2.  **Efecto de Demanda:** La caída en el coste del capital estimula la demanda agregada ($Y^d > Y$) a través de la curva IS. La producción ($Y$) empieza a crecer gradualmente guiada por la velocidad de ajuste $\nu$.
3.  **Mecanismo de Ajuste a Medio Plazo:** A medida que $Y$ supera la renta de pleno empleo $\bar{Y}$, aparece una brecha de producción positiva, lo que presiona los precios al alza (inflación positiva por Curva de Phillips).
4.  **Retorno al Largo Plazo (Neutralidad):** El aumento continuo de precios reduce los saldos monetarios reales ($M - P \downarrow$), desplazando la curva LM de nuevo hacia arriba. El tipo de interés sube y el producto vuelve a contraerse paulatinamente hasta retornar a la capacidad potencial $\bar{Y}$.

### 4.2 El Diagrama de Fases en el Plano $(Y, P)$
Para analizar geométricamente la estabilidad y transición, construimos el diagrama de fases localizando las curvas de demarcación donde las derivadas son nulas:
*   **Locus $\dot{P} = 0$:** Ocurre cuando $Y = \bar{Y}$. Es una línea vertical en el nivel de producto potencial. A la derecha, los precios suben ($\dot{P} > 0$); a la izquierda, bajan ($\dot{P} < 0$).
*   **Locus $\dot{Y} = 0$:** Ocurre cuando la demanda es igual a la producción ($Y^d = Y$). Sustituyendo las ecuacionesLM y Phillips obtenemos una relación lineal decreciente en el plano $(Y, P)$:
    $$P = M + \theta \left( \frac{\beta_0}{\beta_1} - \mu \bar{Y} \right) + \left( \theta \mu - \frac{\theta}{\beta_1} - \psi \right) Y$$
    Por debajo de esta curva, la producción se expande ($\dot{Y} > 0$); por encima, se contrae ($\dot{Y} < 0$).

Utiliza los deslizadores a continuación para simular este comportamiento dinámico de forma interactiva:


In [6]:
# Simulación interactiva con diagrama de fases
@manipulate for m0_val in 80.0:1.0:120.0, beta0_val in 1800.0:10.0:2400.0
    
    params_sim = default_calibration(ISLMParams)
    ss_init = steady_state(params_sim)
    initial_state = [ss_init["Y"], ss_init["P"]]
    
    params_sim = ISLMParams(params_sim.theta, params_sim.psi, params_sim.beta1, 
                            params_sim.mi, params_sim.ni, beta0_val, m0_val, params_sim.ypot0)
    
    t_span = (0.0, 30.0)
    t_eval = collect(range(0.0, 30.0, length=300))
    
    res = simulate_shock(params_sim, initial_state, t_span, t_eval)
    Y_path = res[1, :]
    P_path = res[2, :]
    ss_final = steady_state(params_sim)
    
    # Panel 1: Renta
    p1 = plot(t_eval, Y_path, color=:steelblue, lw=2.5, label="Producción (Y)")
    hline!([params_sim.ypot0], color=:red, ls=:dash, label="Renta Potencial")
    title!("Evolución de la Renta")
    xlabel!("Tiempo (t)")
    ylabel!("Y")
    
    # Panel 2: Precios
    p2 = plot(t_eval, P_path, color=:forestgreen, lw=2.5, label="Precios (P)")
    hline!([ss_init["P"]], color=:gray, ls=:dot, label="P inicial")
    hline!([ss_final["P"]], color=:black, ls=:dash, label="P final")
    title!("Evolución de Precios")
    xlabel!("Tiempo (t)")
    ylabel!("P")
    
    # Panel 3: Diagrama de Fases
    p3 = plot(Y_path, P_path, color=:purple, lw=3, label="Trayectoria dinámica")
    vline!([params_sim.ypot0], color=:orange, ls=:dash, lw=2, label="P_dot = 0")
    
    Y_vals = range(minimum(Y_path)-15, maximum(Y_path)+15, length=100)
    slope = params_sim.theta * params_sim.mi - params_sim.theta / params_sim.beta1 - params_sim.psi
    int_init = ss_init["P"] - slope * params_sim.ypot0
    int_final = ss_final["P"] - slope * params_sim.ypot0
    
    P_loc_init = int_init .+ slope .* Y_vals
    P_loc_final = int_final .+ slope .* Y_vals
    
    plot!(Y_vals, P_loc_init, color=:blue, ls=:dot, label="Y_dot = 0 (Inicial)")
    plot!(Y_vals, P_loc_final, color=:blue, ls=:dash, lw=2, label="Y_dot = 0 (Final)")
    
    # Quiver
    Y_grid = range(minimum(Y_path)-8, maximum(Y_path)+8, length=12)
    P_grid = range(minimum(P_path)-0.8, maximum(P_path)+0.8, length=12)
    
    y_pts = Float64[]
    p_pts = Float64[]
    dy_pts = Float64[]
    dp_pts = Float64[]
    
    for y in Y_grid, p in P_grid
        dy, dp = system_dynamics([y, p], params_sim, 0.0)
        norm = sqrt(dy^2 + dp^2)
        if norm > 0
            push!(y_pts, y)
            push!(p_pts, p)
            push!(dy_pts, (dy/norm)*2.0) # Escalar para visualización
            push!(dp_pts, (dp/norm)*0.2)
        end
    end
    quiver!(y_pts, p_pts, quiver=(dy_pts, dp_pts), color=:gray, alpha=0.5)
    
    scatter!([ss_init["Y"]], [ss_init["P"]], color=:gray, markersize=6, label="EE Inicial")
    scatter!([ss_final["Y"]], [ss_final["P"]], color=:black, markersize=8, marker=:star, label="EE Final")
    
    title!("Diagrama de Fases (Y, P)")
    xlabel!("Producción (Y)")
    ylabel!("Nivel de Precios (P)")
    
    plot(p1, p2, p3, layout=(1,3), size=(1100, 350), 
         plot_title="Respuesta del modelo IS-LM", top_margin=10mm)
end


Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Scope(Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :label), Any["m0_val"], Dict{Symbol, Any}(:className => "interact ", :style => Dict{Any, Any}(:padding => "5px 10px 0px 10px")))], Dict{Symbol, Any}(:className => "interact-flex-row-left")), Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :input), Any[], Dict{Symbol, Any}(:max => 41, :min => 1, :attributes => Dict{Any, Any}(:type => "range", Symbol("data-bind") => "numericValue: index, valueUpdate: 'input', event: {change: function (){this.changes(this.changes()+1)}}", "orient" => "horizontal"), :step => 1, :className => "slider slider is-fullwidth", :style => Dict{Any, Any}()))], Dict{Symbol, Any}(:className => "interact-flex-row-center")), Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :p), Any[], Dict{Symbol, Any}(:attributes => Dict("data-bind" => "text: formatted_val")))], Dict{Symbol, Any}(:className => "interact-flex-row-right"))], Dict{Symbol, Any}(:className => "interact-flex-row interact-widget")), Dict{String, Tuple{AbstractObservable, Union{Nothing, Bool}}}("changes" => (Observable(0), nothing), "index" => (Observable{Any}(21), nothing)), Set{String}(), nothing, Asset[Asset("js", "knockout", "C:\\Users\\AntonioRC\\.julia\\packages\\Knockout\\HReiN\\src\\..\\assets\\knockout.js"), Asset("js", "knockout_punches", "C:\\Users\\AntonioRC\\.julia\\packages\\Knockout\\HReiN\\src\\..\\assets\\knockout_punches.js"), Asset("js", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\InteractBase\\8TTmI\\src\\..\\assets\\all.js"), Asset("css", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\InteractBase\\8TTmI\\src\\..\\assets\\style.css"), Asset("css", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\Interact\\PENUy\\src\\..\\assets\\bulma_confined.min.css")], Dict{Any, Any}("changes" => Any[WebIO.JSString("(function (val){return (val!=this.model[\"changes\"]()) ? (this.valueFromJulia[\"changes\"]=true, this.model[\"changes\"](val)) : undefined})")], "index" => Any[WebIO.JSString("(function (val){return (val!=this.model[\"index\"]()) ? (this.valueFromJulia[\"index\"]=true, this.model[\"index\"](val)) : undefined})")]), WebIO.ConnectionPool(Channel{Any}(32), Set{AbstractConnection}(), Base.GenericCondition(ReentrantLock())), WebIO.JSString[WebIO.JSString("function () {\n    var handler = (function (ko, koPunches) {\n    ko.punches.enableAll();\n    ko.bindingHandlers.numericValue = {\n        init: function(element, valueAccessor, allBindings, data, context) {\n            var stringified = ko.observable(ko.unwrap(valueAccessor()));\n            stringified.subscribe(function(value) {\n                var val = parseFloat(value);\n                if (!isNaN(val)) {\n                    valueAccessor()(val);\n                }\n            });\n            valueAccessor().subscribe(function(value) {\n                var str = JSON.stringify(value);\n                if ((str == \"0\") && ([\"-0\", \"-0.\"].indexOf(stringified()) >= 0))\n                     return;\n                 if ([\"null\", \"\"].indexOf(str) >= 0)\n                     return;\n                stringified(str);\n            });\n            ko.applyBindingsToNode(\n                element,\n                {\n                    value: stringified,\n                    valueUpdate: allBindings.get('valueUpdate'),\n                },\n                context,\n            );\n        }\n    };\n    var json_data = {\"formatted_vals\":[\"80.0\",\"81.0\",\"82.0\",\"83.0\",\"84.0\",\"85.0\",\"86.0\",\"87.0\",\"88.0\",\"89.0\",\"90.0\",\"91.0\",\"92.0\",\"93.0\",\"94.0\",\"95.0\",\"96.0\",\"97.0\",\"98.0\",\"99.0\",\"100.0\",\"101.0\",\"102.0\",\"103.0\",\"104.0\",\"105.0\",\"106.0\",\"107.0\",\"108.0\",\"109.0\",\"110.0\",\"111.0\",\"112.0\",\"113.0\",\"114.0\",\"115.0\",\"116.0\",\"117.0\"

## 5. Verificación Numérica contra el Oráculo (Libro)

Para certificar la robustez científica de la simulación, validamos nuestros resultados frente al **Oráculo del Libro** (Apéndices D y E, resueltos originalmente en MATLAB y DYNARE).

| Variable | Oráculo MATLAB / DYNARE | Simulación Python | Estado |
| :--- | :---: | :---: | :---: |
| **Producción de Equilibrio ($Y^*$)** | 2000.00 | 2000.00 | ✅ Verificado (tolerancia < 1e-6) |
| **Nivel de Precios de Equilibrio ($P^*$)** | 81.00 | 81.00 | ✅ Verificado (tolerancia < 1e-6) |
| **Tipo de Interés de Equilibrio ($i^*$)** | 2.00% | 2.00% | ✅ Verificado (tolerancia < 1e-6) |

Esta validación cruzada garantiza que las diferencias temporales y dinámicas provienen exclusivamente de la aproximación de tiempo continuo frente a las aproximaciones discretas, sin errores conceptuales en las ecuaciones.


In [7]:
@assert isapprox(ss["Y"], 2000.0; atol=1e-5)
@assert isapprox(ss["P"], 81.0; atol=1e-5)
@assert isapprox(ss["i"], 2.0; atol=1e-5)
println("OK: coincide con el oráculo.")


OK: coincide con el oráculo.


## 6. Buenas Prácticas Aplicadas en este Laboratorio

Fíjate en las siguientes decisiones de diseño técnico tomadas para estructurar esta práctica de forma ejemplar:
1.  **Código Modular**: La lógica matemática de las ecuaciones del modelo y el solucionador numérico no están en este notebook. Están aislados en el archivo modular `src/macroaicomp/models/islm.py` para asegurar su reutilización.
2.  **Calibración Aislada**: No hay números "mágicos" embebidos en el cálculo. Los parámetros se cargan desde un diccionario o estructura al principio.
3.  **Independencia Didáctica**: El notebook funciona de manera autónoma como un *frontend* interactivo sin saturar al alumno con detalles de codificación de bajo nivel.


## 7. Cuaderno de Bitácora (Actividades para el Alumno)

Responde en tu Cuaderno de Bitácora digital las siguientes preguntas basándote en tus observaciones de la simulación:

1.  **El Shock Monetario:** Deja el *Gasto Autónomo* ($B_0$) en su nivel base ($2100.0$) e incrementa la *Oferta Monetaria* ($M_0$) a $115.0$.
    *   ¿Qué ocurre con la Renta ($Y$) a corto plazo? ¿Por qué?
    *   ¿Qué ocurre con el nivel de precios ($P$) a largo plazo? Explica la relación porcentual entre el aumento de dinero y el aumento de precios en el equilibrio final.
    *   ¿Se cumple el principio de *neutralidad del dinero*? Justifica tu respuesta.
2.  **El Shock Fiscal:** Restaura $M_0$ a $100.0$ e incrementa el *Gasto Autónomo* a $2200.0$ (shock de política fiscal expansiva).
    *   Describe la trayectoria de la Renta ($Y$). ¿Aumenta de forma permanente o transitoria?
    *   ¿Qué ocurre con el tipo de interés nominal ($i$) a largo plazo? Explica el fenómeno económico conocido como *efecto expulsión* (crowding-out) a partir de los resultados y el gráfico.
3.  **Análisis del Diagrama de Fases (Espacio de Estados):**
    *   Observa la trayectoria de transición en el plano $(Y, P)$ en el Panel 3. ¿Por qué la trayectoria se inicia con un movimiento puramente horizontal hacia la derecha y no en diagonal desde el principio? Relaciona esto con el supuesto de rigidez instantánea de los precios ($P$) y el ajuste gradual de la renta ($Y$) ante excesos de demanda.
    *   Explica por qué la trayectoria cruza el locus $\dot{Y}=0$ con una tangente vertical. ¿Qué valor toma la derivada $\dot{Y}$ en ese instante exacto de cruce?
    *   ¿Cómo cambia la trayectoria en el diagrama si la velocidad de ajuste del mercado de bienes ($\nu$) disminuye drásticamente a $0.05$? Compárala con el comportamiento bajo la calibración original ($\nu=0.2$).


## 8. Benchmark de Rendimiento (Fase III)
Evaluamos la velocidad de simulación usando `BenchmarkTools.jl`.

In [8]:
# Benchmark simulation
@btime simulate_shock($params, [2000.0, 81.0], (0.0, 50.0), collect(range(0.0, 50.0, length=500)))


  121.000 μs (15624 allocations: 632.32 KiB)


retcode: Success
Interpolation: 1st order linear
t: 500-element Vector{Float64}:
  0.0
  0.10020040080160321
  0.20040080160320642
  0.30060120240480964
  0.40080160320641284
  0.501002004008016
  0.6012024048096193
  0.7014028056112225
  0.8016032064128257
  0.9018036072144289
  1.002004008016032
  1.1022044088176353
  1.2024048096192386
  ⋮
 48.897795591182366
 48.99799599198397
 49.098196392785574
 49.19839679358717
 49.298597194388776
 49.39879759519038
 49.498997995991985
 49.59919839679359
 49.699398797595194
 49.79959919839679
 49.899799599198396
 50.0
u: 500-element Vector{Vector{Float64}}:
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 ⋮
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.0, 81.0]
 [2000.